# 03 · Block-level regression (milestone M3)

LST ~ material fractions + vegetation + built form, fitted on urban blocks of the model area. R² is reported with **spatial** cross-validation (1 km tiles held out together), because neighbouring blocks share heat and random folds would overstate accuracy.

All temperatures are **land surface temperature**, not air temperature.

**Cool roofs** use a literature coefficient (Wang, Huang & Li 2020, GRL; sources in `config/aoi.yaml`) because the interim Sentinel-2 regression gives dark roofs a cooling sign, likely from building shadow. **Trees** use the fitted coefficients.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from coolcairo.config import load_config, geobox_for
cfg = load_config()

In [ ]:
from coolcairo.pipeline import build_blocks
from coolcairo.model import fit, training_rows, FEATURES
blocks, material, buildings = build_blocks(cfg)
train = training_rows(blocks, cfg['min_building_coverage'])
result = fit(train, cfg['cv_tile_size'], cfg['cv_folds'])
result.to_dict()

In [ ]:
from coolcairo.interventions import full_adoption_summary, plausibility_warnings, cool_roof_surface_delta
print(plausibility_warnings(cfg, result) or 'Coefficient signs plausible.')
cr = cfg['cool_roof']
print(f"Cool roof ({cr['method']}): {cool_roof_surface_delta(cfg, result):+.2f} °C surface temperature per unit block area converted")
print(f"  = {cr['dts_per_albedo']} K per unit albedo x ({cr['albedo_after']} - {cr['albedo_before']})")
full_adoption_summary(cfg, result, train).round(2)

In [ ]:
import numpy as np
pred = result.intercept + train[FEATURES].to_numpy() @ np.array([result.coefficients[f] for f in FEATURES])
plt.figure(figsize=(6, 6))
plt.scatter(train.lst_c, pred, s=3, alpha=0.4)
lo, hi = train.lst_c.min(), train.lst_c.max()
plt.plot([lo, hi], [lo, hi], 'k--', lw=1)
plt.xlabel('Observed LST (°C)'); plt.ylabel('Fitted LST (°C)')
plt.title(f"Spatial-CV R² = {result.r2_spatial_cv:.2f}")